# Logistic Regression Baseline: German Credit
DSE4212/QF4212 Group Project 1

Convention used everywhere: **positive class (1) = default / bad credit**.

- Part 1: Naive baseline (the 7-feature model from the Chapter 6 notebook)
- Part 2: Reference model (untuned logistic regression on the shared pipeline)
- Part 3: Hyperparameter tuning (cross-validation on training data only)
- Part 4: Test-set evaluation and operating points (thresholds)
- Part 5: Interpretation (coefficients and odds ratios)
- Part 6: Save model, probabilities and metrics

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict

# Find the project root (the folder that contains src/) so `from src...` works
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import make_pipeline, check_inputs   # shared preprocessing (teammate's module)
from src.evaluation import (evaluate, metrics_only, results_table,
                            f1_optimal_threshold, cost_min_threshold,
                            cost_optimal_threshold, bootstrap_auc_ci, RANDOM_STATE)

matplotlib.rcParams.update({'font.size': 12})

# Output folders
OUT = ROOT / "outputs"
for sub in ("models", "predictions", "results"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)

# Settings
TARGET = "credit_risk"   # 1 = default (bad), 0 = good
FN_COST = 5              # a missed default costs 5x a wrongly rejected good client

## Load Data and inspect
The team's `train.csv` and `test.csv` are used as they are (800 / 200 rows), so every model
in the project is trained and tested on identical rows.

In [ ]:
train = pd.read_csv(OUT / "train.csv")
test = pd.read_csv(OUT / "test.csv")
print("train:", train.shape, "| test:", test.shape)
train.head()

In [ ]:
# Separate features and target
y_train, y_test = train[TARGET].astype(int), test[TARGET].astype(int)
X_train, X_test = train.drop(columns=[TARGET]), test.drop(columns=[TARGET])

print(f"Default rate: train {y_train.mean():.3f}, test {y_test.mean():.3f}")   # German Credit has ~30% bad
check_inputs(X_train)   # raises if expected columns are missing

## Part 1: Naive baseline (Chapter 6 approach)
In class we fitted a plain `LogisticRegression()` on 7 numeric columns only, with no scaling,
no categorical variables and no tuning. We repeat it here on the **shared** train/test split so
it is directly comparable with everything below.

In [ ]:
naive_features = ["log1p_credit_amount", "log1p_duration_months", "installment_rate",
                  "residence_since", "age_years", "existing_credits", "people_liable"]

naive = LogisticRegression(max_iter=1000)
naive.fit(X_train[naive_features], y_train)

res_naive = evaluate(naive, X_test[naive_features], y_test,
                     name="Naive LR (7 numeric features)", fn_cost=FN_COST, report=True)

**Remark.** Accuracy looks acceptable only because 70% of clients are good: predicting
"good" for everyone already scores 0.70. The recall on defaults and the AUC are the
informative numbers, and both are weak. The remaining parts improve this step by step.

## Part 2: Reference model (untuned logistic regression + shared pipeline)
`make_pipeline` (in `src/preprocessing.py`) one-hot encodes the categorical columns, scales the
numeric ones and adds the engineered flags. The classifier is the team's untuned
`LogisticRegression(max_iter=1000)` (C = 1, L2 penalty).

In [ ]:
lr_default = make_pipeline(LogisticRegression(max_iter=1000), scale_numeric=True)
res_default = evaluate(lr_default, X_test, y_test, name="LR untuned (C=1, L2)", fn_cost=FN_COST)

## Part 3: Hyperparameter tuning (training data only)
`RepeatedStratifiedKFold` (5 folds x 2 repeats) on the training set. The scoring metric is
ROC AUC, which does not depend on a threshold and is not distorted by class imbalance.

Grid: `C` (inverse regularisation strength, 11 log-spaced values) x `l1_ratio`
(0 = ridge, 0.5 = elastic net, 1 = lasso). The SAGA solver supports all three.
Requires scikit-learn >= 1.8.

In [ ]:
base = make_pipeline(
    LogisticRegression(solver="saga", l1_ratio=0, max_iter=20000, random_state=RANDOM_STATE),
    scale_numeric=True,
)
param_grid = {
    "classifier__C": np.logspace(-3, 2, 11),
    "classifier__l1_ratio": [0.0, 0.5, 1.0],
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
gs = GridSearchCV(base, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1, refit=True)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    gs.fit(X_train, y_train)

best = gs.best_estimator_
cvr = pd.DataFrame(gs.cv_results_).sort_values("rank_test_score")
cv_std = cvr.iloc[0]["std_test_score"]

print(f"Best params: {gs.best_params_}")
print(f"CV AUC: {gs.best_score_:.4f} (+/- {cv_std:.4f})")
cvr[["param_classifier__C", "param_classifier__l1_ratio", "mean_test_score", "std_test_score"]].head(8)

In [ ]:
# CV AUC across the grid
fig, ax = plt.subplots(figsize=(7, 4.5))
names = {0.0: "L2 (ridge)", 0.5: "elastic net", 1.0: "L1 (lasso)"}
for l1, g in cvr.groupby("param_classifier__l1_ratio"):
    g = g.sort_values("param_classifier__C")
    ax.plot(g["param_classifier__C"].astype(float), g["mean_test_score"],
            marker="o", label=names[float(l1)])
ax.set_xscale("log")
ax.set_xlabel("C (inverse regularisation strength)")
ax.set_ylabel("CV ROC AUC")
ax.set_title("Logistic regression: tuning curve")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig(OUT / "results/logreg_tuning_curve.png", dpi=200)
plt.show()

**Remark.** Strong regularisation (small C) with lasso or elastic net shrinks everything
to zero and the AUC collapses to 0.5. Ridge is the most stable choice, and the AUC is flat
for moderate to large C, so tuning gives only a small gain over the untuned model.

## Part 4: Test-set evaluation and operating points
The AUC measures *ranking* quality. The classification **threshold** is a separate,
cost-driven decision, so it is chosen from **out-of-fold training predictions**
(never on the test set):

- 0.5, the default
- the F1-optimal threshold
- the cost-optimal threshold, where a missed default costs `FN_COST` times a wrongly rejected good client

In [ ]:
# Out-of-fold probabilities on the training set
oof = cross_val_predict(best, X_train, y_train, method="predict_proba",
                        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))[:, 1]
pd.DataFrame({"index": X_train.index, "y_true": y_train.values, "y_proba": oof}).to_csv(
    OUT / "predictions/logreg_train_oof_proba.csv", index=False)

t_f1 = f1_optimal_threshold(y_train, oof)
t_cost = cost_min_threshold(y_train, oof, cost_fn=FN_COST)
print(f"F1-optimal threshold: {t_f1:.3f} | cost-optimal threshold (FN:FP = {FN_COST}:1): {t_cost:.3f} "
      f"(theory for perfectly calibrated probabilities: {cost_optimal_threshold(FN_COST):.3f})")

In [ ]:
res_05 = evaluate(best, X_test, y_test, threshold=0.5, name="LR tuned @ 0.5", fn_cost=FN_COST,
                  save_to=str(OUT / "predictions/logreg_test_proba.csv"))
res_f1 = evaluate(best, X_test, y_test, threshold=t_f1, name="LR tuned @ F1-optimal", fn_cost=FN_COST)
res_cost = evaluate(best, X_test, y_test, threshold=t_cost, name="LR tuned @ cost-optimal", fn_cost=FN_COST)

ci = bootstrap_auc_ci(y_test, res_05["y_proba"])
print(f"\nTest AUC = {res_05['auc']:.3f}, 95% bootstrap CI [{ci[0]:.3f}, {ci[1]:.3f}]")

In [ ]:
table = results_table([res_naive, res_default, res_05, res_f1, res_cost], fn_cost=FN_COST)
table.to_csv(OUT / "results/logreg_results_table.csv", index=False)
table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC curves on the test set
for res, label in [(res_naive, "naive"), (res_default, "untuned"), (res_05, "tuned")]:
    fpr, tpr, _ = roc_curve(y_test, res["y_proba"])
    axes[0].plot(fpr, tpr, label=f"{label} (AUC = {res['auc']:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC - test set")
axes[0].legend(loc="lower right"); axes[0].grid(alpha=0.3)

# Confusion matrix of the tuned model at the cost-optimal threshold
ConfusionMatrixDisplay(res_cost["confusion_matrix"], display_labels=["good", "default"]).plot(
    ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title(f"Confusion matrix (tuned, threshold {t_cost:.3f})")

plt.tight_layout()
plt.savefig(OUT / "results/logreg_roc_confusion.png", dpi=200)
plt.show()

**Remark.** Tuning barely changes the AUC, but the threshold changes the recall on defaults
a lot. Lowering the threshold catches more defaulters at the price of rejecting more good
clients. Which operating point is right depends on the cost ratio, which is the key input
for the business recommendations.

## Part 5: Interpretation (coefficients)
Numeric features are standardised, so their coefficients are the change in log-odds of
default per one standard deviation. One-hot categories and engineered 0/1 flags are not
scaled: their coefficients compare the category with its absence. `odds_ratio = exp(coef)`.

In [ ]:
feat_names = [n.split("__", 1)[-1] for n in best.named_steps["preprocessing"].get_feature_names_out()]
coefs = pd.DataFrame({"feature": feat_names, "coef": best.named_steps["classifier"].coef_.ravel()})
coefs["odds_ratio"] = np.exp(coefs["coef"])
coefs = coefs.reindex(coefs["coef"].abs().sort_values(ascending=False).index).reset_index(drop=True)
coefs.to_csv(OUT / "results/logreg_coefficients.csv", index=False)
print(f"{(coefs['coef'] == 0).sum()} of {len(coefs)} coefficients shrunk exactly to zero")

top = coefs.head(15).iloc[::-1]
plt.figure(figsize=(8, 6))
plt.barh(top["feature"], top["coef"], color=np.where(top["coef"] > 0, "firebrick", "steelblue"))
plt.axvline(0, color="k", lw=0.8)
plt.xlabel("Coefficient (log-odds of default)")
plt.title("Logistic regression: 15 largest coefficients\n(red = raises default risk)")
plt.tight_layout()
plt.savefig(OUT / "results/logreg_coefficients.png", dpi=200)
plt.show()
coefs.head(15)

## Part 6: Save model, probabilities and metrics

In [ ]:
joblib.dump(best, OUT / "models/logreg_best.joblib")

summary = {
    "best_params": {k.replace("classifier__", ""): float(v) for k, v in gs.best_params_.items()},
    "cv_auc_mean": float(gs.best_score_), "cv_auc_std": float(cv_std),
    "test_auc_ci95": ci, "fn_cost": FN_COST,
    "thresholds": {"default": 0.5, "f1_opt": t_f1, "cost_opt": t_cost},
    "results": {r["name"]: metrics_only(r)
                for r in (res_naive, res_default, res_05, res_f1, res_cost)},
}
(OUT / "results/logreg_metrics.json").write_text(json.dumps(summary, indent=2))
print("Saved to", OUT)

## Notes for the report

- **For teammates (RF, AdaBoost):** call
  `evaluate(model, X_test, y_test, name=..., save_to="outputs/predictions/<model>_test_proba.csv")`
  so all models produce identical metrics and a probability file that can be overlaid on one ROC plot.
- The test set is small (about 200 customers), so compare models with the AUC confidence interval in mind.
- Thresholds were set on out-of-fold training predictions. The test set was used once, for the final numbers.
- `gender_group` and `age_group` are sensitive attributes. Keep them out of the model inputs
  (check what `make_pipeline` does) and use them only for the optional fairness analysis.